In [ ]:
# Actividad: Introducción a la Ingeniería de Datos
## Tema: Diagnóstico por imágenes para el cálculo de SINS (Spinal Instability Neoplastic Score)

In [ ]:
## 1. Prompt utilizado para generar el script
"Actúa como un científico de datos experto en visión computacional... [copia el prompt exacto que usaste]"

In [ ]:
# -*- coding: utf-8 -*-
"""Script para descargar y explorar el dataset spine_frx_normal_vindr desde Roboflow Universe.
   Tema: Diagnóstico por imágenes para cálculo de SINS
   ¡ATENCIÓN! Este script contiene una API KEY. NO lo compartas públicamente.
"""

# ============================================================
# 1. INSTALACIÓN DE DEPENDENCIAS
# ============================================================
!pip install roboflow pandas matplotlib opencv-python-headless -q

# ============================================================
# 2. IMPORTACIÓN DE LIBRERÍAS
# ============================================================
import os
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import json
from roboflow import Roboflow

# ============================================================
# 3. CONFIGURACIÓN (API KEY YA INSERTADA)
# ============================================================
# ⚠️ IMPORTANTE: Esta clave está expuesta. Revocarla en Roboflow y generar una nueva.
API_KEY = "6j2JJI8Qpk8qhYs5mIZq"  # <--- REEMPLAZAR por tu NUEVA clave después de revocar esta

# 📦 URL del dataset en Roboflow Universe
DATASET_URL = "https://universe.roboflow.com/spine1000-fqdvf/spine_frx_normal_vindr"
VERSION = 1  # Número de versión del dataset

# ============================================================
# 4. FUNCIONES MODULARIZADAS
# ============================================================

def descargar_dataset(api_key: str, dataset_url: str, version: int = 1) -> str:
    """
    Descarga un dataset desde Roboflow Universe usando la API key.

    Args:
        api_key (str): Clave API de Roboflow.
        dataset_url (str): URL completa del dataset en Roboflow Universe.
        version (int): Número de versión del dataset.

    Returns:
        str: Ruta local donde se descargó el dataset.
    """
    rf = Roboflow(api_key=api_key)
    # Extraer el workspace y project name de la URL
    # Formato: https://universe.roboflow.com/workspace/project
    parts = dataset_url.rstrip('/').split('/')
    workspace = parts[-2]
    project_name = parts[-1]

    project = rf.workspace(workspace).project(project_name)
    dataset = project.version(version).download("yolov8")
    print(f"✅ Dataset descargado en: {dataset.location}")
    return dataset.location

def cargar_anotaciones(ruta_dataset: str) -> pd.DataFrame:
    """
    Carga las anotaciones del dataset desde el archivo _annotations.coco.json.

    Args:
        ruta_dataset (str): Ruta donde se encuentra el dataset descargado.

    Returns:
        pd.DataFrame: DataFrame con las anotaciones (imagen, clase, bbox).
    """
    # Buscar el archivo de anotaciones COCO
    annot_path = os.path.join(ruta_dataset, "_annotations.coco.json")
    if not os.path.exists(annot_path):
        possible_files = ["_annotations.json", "annotations.json"]
        for f in possible_files:
            test_path = os.path.join(ruta_dataset, f)
            if os.path.exists(test_path):
                annot_path = test_path
                break
        else:
            raise FileNotFoundError(f"No se encontró archivo de anotaciones en {ruta_dataset}")

    with open(annot_path, "r") as f:
        data = json.load(f)

    # Extraer información de imágenes y categorías
    images = {img["id"]: img["file_name"] for img in data["images"]}
    categories = {cat["id"]: cat["name"] for cat in data["categories"]}

    rows = []
    for ann in data["annotations"]:
        rows.append({
            "image_id": ann["image_id"],
            "image_file": images.get(ann["image_id"], "desconocido"),
            "category_id": ann["category_id"],
            "category_name": categories.get(ann["category_id"], "desconocido"),
            "bbox_x": ann["bbox"][0],
            "bbox_y": ann["bbox"][1],
            "bbox_w": ann["bbox"][2],
            "bbox_h": ann["bbox"][3],
            "area": ann["area"],
        })

    df = pd.DataFrame(rows)
    print(f"📊 Anotaciones cargadas: {len(df)} registros")
    return df

def visualizar_ejemplo(df: pd.DataFrame, ruta_dataset: str, idx: int = 0) -> None:
    """
    Muestra una imagen del dataset con su anotación dibujada.

    Args:
        df (pd.DataFrame): DataFrame con las anotaciones.
        ruta_dataset (str): Ruta donde se encuentra el dataset.
        idx (int): Índice de la fila a visualizar.
    """
    if df.empty:
        print("⚠️ No hay datos para visualizar.")
        return

    row = df.iloc[idx]
    img_path = os.path.join(ruta_dataset, row["image_file"])

    if not os.path.exists(img_path):
        print(f"⚠️ Imagen no encontrada: {img_path}")
        return

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    x, y, w, h = int(row["bbox_x"]), int(row["bbox_y"]), int(row["bbox_w"]), int(row["bbox_h"])
    cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.putText(img, row["category_name"], (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.title(f"Imagen: {row['image_file']} | Clase: {row['category_name']}")
    plt.axis("off")
    plt.show()

def resumen_dataset(df: pd.DataFrame) -> None:
    """
    Imprime un resumen estadístico del dataset.

    Args:
        df (pd.DataFrame): DataFrame con las anotaciones.
    """
    print("\n" + "="*50)
    print("📊 RESUMEN DEL DATASET")
    print("="*50)
    print(f"Total de anotaciones: {len(df)}")
    print(f"Total de imágenes únicas: {df['image_id'].nunique()}")
    print("\n📌 Distribución de clases:")
    print(df["category_name"].value_counts())
    print("\n📌 Verificación de valores nulos:")
    print(df.isnull().sum())
    print("\n📌 Primeras 5 filas:")
    print(df.head())

def main():
    """
    Función principal que orquesta todo el flujo de trabajo.
    """
    print("🚀 INICIANDO PIPELINE DE DATOS PARA SINS")
    print("="*50)

    # Paso 1: Descargar dataset
    print("\n📥 Descargando dataset desde Roboflow...")
    ruta = descargar_dataset(API_KEY, DATASET_URL, VERSION)

    # Paso 2: Cargar anotaciones
    print("\n📂 Cargando anotaciones...")
    df = cargar_anotaciones(ruta)

    # Paso 3: Resumen del dataset
    resumen_dataset(df)

    # Paso 4: Visualizar un ejemplo
    print("\n🖼️ Visualizando ejemplo...")
    visualizar_ejemplo(df, ruta, idx=0)

    print("\n✅ Pipeline completado exitosamente.")

# ============================================================
# 5. EJECUCIÓN PRINCIPAL
# ============================================================
if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.5 MB/s eta 0:00:00
🚀 INICIANDO PIPELINE DE DATOS PARA SINS

📥 Descargando dataset desde Roboflow...
loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov8 in progress : 0.2%

In [ ]:
## 2. Identificación de elementos en el script

| Elemento | Tipo / Explicación |
| :--- | :--- |
| `import pandas as pd` | Carga el módulo pandas con el alias `pd`. |
| `from roboflow import Roboflow` | Importa la clase `Roboflow` para usar la API. |
| `API_KEY = "..."` | Variable de tipo string que almacena la clave de autenticación. |
| `Roboflow(api_key)` | Método constructor que inicializa la conexión con Roboflow. |
| `project.version(1).download("yolov8")` | Método encadenado que descarga el dataset. |
| `pd.DataFrame(rows)` | Constructor de DataFrame a partir de una lista de diccionarios. |
| `df["category_name"].value_counts()` | Método que cuenta la frecuencia de cada categoría. |
| `def descargar_dataset(...):` | Definición de función con parámetros y docstring. |

In [ ]:
## 3. Refactorización del código (KISS + Docstrings)

Para mejorar el código, se le pidió al LLM que lo modularizara en funciones, siguiendo el principio KISS (Keep It Simple, Stupid). El objetivo era que cada función hiciera una sola cosa y fuera fácil de entender.

**Funciones creadas:**
*   `descargar_dataset()`: Se encarga de la descarga del dataset.
*   `cargar_anotaciones()`: Carga las anotaciones en un DataFrame.
*   `visualizar_ejemplo()`: Muestra una imagen con su anotación.
*   `resumen_dataset()`: Imprime estadísticas del dataset.
*   `main()`: Orquesta el flujo principal del programa.

Además, se agregaron **docstrings** a cada función para documentar su propósito, parámetros y valor de retorno.

In [ ]:
## 4. Resultados de la ejecución

Al ejecutar el script, se obtiene lo siguiente:
1.  Se descarga el dataset desde Roboflow.
2.  Se cargan todas las anotaciones en un DataFrame.
3.  Se muestra un resumen con el total de imágenes, la distribución de clases y la verificación de valores nulos.
4.  Se visualiza una imagen de ejemplo con su caja delimitadora (bounding box).